In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.metrics import *
from tqdm import tqdm


In [ ]:
dataset = 'mdcd_3yrs'
model_name = 'transformer_fullhistory_6mopred'
data_path = f'/data2/processed_datasets/ak4885/psychosis_schizophrenia_prediction/raw_data_{dataset}'
int_data_path = f'/data2/processed_datasets/ak4885/psychosis_schizophrenia_prediction/intermediate_data_{dataset}'
model_path = f'../skip_model_training/models/{model_name}'
featuretype = 'fullfeatures'
output_folder = f'figures_skiptrain_6mo_{featuretype}/'

test_output = pd.read_csv(f'{output_folder}/test_outputs.csv')
test_output = test_output.loc[test_output['days_since_start'] >= 365]

In [ ]:
minmax_tte = test_output.groupby('person_id').agg({'tte':['min', 'max']})
minmax_tte.columns = ['min', 'max']
long_pred_trajectory_pids = minmax_tte.loc[(minmax_tte['max']-minmax_tte['min'])/365 > 2.5].index

In [ ]:
print(len(test_output))
test_output = test_output.loc[test_output['person_id'].isin(long_pred_trajectory_pids)]
print(len(test_output))

In [ ]:
def evaluate_metrics(y_true, y_pred, cutoff_prob, dict_metrics, binary_funcs, y_pred_binary = None):
    """
    Evaluate a list of metrics for the given predictions and true values.

    Parameters:
    y_pred (array-like): Predicted values
    y_true (array-like): True values
    cutoff (float): Cutoff value for binary classification
    dict_metrics (dict): Dictionary of metric names and their corresponding functions
    binary_funcs (list): list of function names that take in y_pred_binary instead of y_pred (probability)

    Returns:
    dict: Dictionary of metric results
    """
    
    results = {}
    for metric_name, metric_func in dict_metrics.items():
        if metric_name in binary_funcs:
            results[metric_name] = metric_func(y_true, y_pred_binary)
        else: 
            results[metric_name] = metric_func(y_true, y_pred)
    
    return results

def bootstrap_evaluation(y_true, y_pred, cutoff_prob, dict_metrics, binary_funcs, n_bootstraps = 300, alpha = 0.05, y_pred_binary = None):
    #alpha = 0.05 gives us 95% CI
    # make y_true and y_pred arrays
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    # get sampling
    rng = np.random.RandomState(seed=44)
    idx = np.arange(y_true.shape[0])

    # get blank dictionary for results
    metrics_results = {metric: [] for metric in dict_metrics.keys()}

    for i in tqdm(range(0, n_bootstraps)):
        subsample_idx = rng.choice(idx, size=idx.shape[0], replace=True)
        y_true_subsample = y_true[subsample_idx]
        y_pred_subsample = y_pred[subsample_idx]

        
        if len(set(y_true_subsample)) > 1:
            subsample_results = evaluate_metrics(y_true_subsample, y_pred_subsample, cutoff_prob, dict_metrics, binary_funcs, None)

            for metric_name, metric_value in subsample_results.items():
                metrics_results[metric_name].append(metric_value)
                
    return metrics_results

def get_summary_stats(metrics_results, alpha):            
    # get summary stats
    summary_stats = {}
    for metric_name, values in metrics_results.items():
        mean = np.mean(values)
        ci_low = np.percentile(values, 100 * alpha / 2)
        ci_high = np.percentile(values, 100 * (1 - alpha / 2))
        
        summary_stats[metric_name] = {
            'mean': mean,
            'CI_low': ci_low,
            'CI_high': ci_high
        }
    
    summary_df = pd.DataFrame(summary_stats).T
    return summary_df

list_times = np.linspace(0, 1080, 12)
column = 'days_since_psychosis'
list_eval = []
metric_functions = {
    'AUROC': roc_auc_score,
    'Brier': brier_score_loss}
binary_metrics = []

for i in range(1, len(list_times)):
    temp_outputs = test_output.loc[(test_output[column] <= list_times[i]) & (test_output[column] > list_times[i-1])]
    print(len(temp_outputs))
    boot_eval = bootstrap_evaluation(temp_outputs['y_true'], temp_outputs['y_pred'], None, metric_functions, binary_metrics, y_pred_binary = 'y_pred_binary')
    summary_eval = get_summary_stats(boot_eval, 0.95)
    list_eval.append(summary_eval)



In [ ]:
metric = 'AUROC'
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(9,7))
font = {'weight' : 'bold',
        'size'   : 24}
matplotlib.rc('font', **font)
x_label_coord = -0
y_label_coord = 1.02

plt.subplot(1,1,1)
matplotlib.rc('font', **font)
plt.plot(list_times[0:-1]/365, [df.loc[metric, "mean"] for df in list_eval], c = 'k')
plt.fill_between(list_times[0:-1]/365, [df.loc[metric, "CI_low"] for df in list_eval], [df.loc[metric, "CI_high"] for df in list_eval], color = 'k', alpha=0.5)
plt.xlabel('Years since psychosis', fontdict=font)
plt.ylabel('AUROC', fontdict=font)
plt.xticks(fontsize=font["size"], fontweight=font["weight"])
plt.yticks(fontsize=font["size"], fontweight=font["weight"])
plt.title('Overall performance')
ax.text(x_label_coord, y_label_coord, 'A', transform=ax.transAxes, size=24, weight='bold')
plt.ylim([0.7, 0.8])


# plot number of patients and scz prevalence as a function of time since psychosis

In [ ]:
num_days_prediction = 90
df_pop = pd.read_csv(f'{data_path}/population_2dx.csv', parse_dates = ['psychosis_diagnosis_date', 'scz_diagnosis_date', 'cohort_start_date'])
print(len(df_pop), df_pop['sz_flag'].sum()/len(df_pop), len(df_pop['person_id'].unique()))
df_pop = df_pop.loc[(df_pop['cohort_start_date']-df_pop['psychosis_diagnosis_date']).dt.days >= num_days_prediction]
print(len(df_pop), df_pop['sz_flag'].sum()/len(df_pop), len(df_pop['person_id'].unique()))
df_pop['censor_date'] = df_pop['cohort_start_date'] - pd.Timedelta(days=num_days_prediction)


In [ ]:
df_pop['days_psychosis_to_censor'] = (df_pop['censor_date']-df_pop['psychosis_diagnosis_date']).dt.days

# range of days --> years between psychosis and censor date
list_percentiles = np.linspace(1,100, 100)
percentile_days = np.percentile(df_pop['days_psychosis_to_censor'], [list_percentiles]).reshape(-1)
percentile_years = percentile_days/365

# prevalence of schizophrenia as a function of time since psychosis
prevalences = []
total_n_patients = []
for days in percentile_days:
    temp_df = df_pop.loc[df_pop['days_psychosis_to_censor'] >= days]
    prevalences.append(100*sum(temp_df['sz_flag'])/len(temp_df))
    total_n_patients.append(len(temp_df))


temp_df = df_pop.loc[df_pop['days_psychosis_to_censor'] >= 90*12]

# plotting
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(20,7))
font = {'weight' : 'bold',
        'size'   : 24}
matplotlib.rc('font', **font)
x_label_coord = -0
y_label_coord = 1.02

ax = axes[0]
ax.plot(percentile_years, prevalences)
ax.vlines(2.958, 0, 8.5, color = 'red', label = f'Cutoff prevalence: {np.round((100*sum(temp_df["sz_flag"])/len(temp_df)), 2)}%')
ax.set_xlabel('Years since psychosis diagnosis', fontweight='bold')
ax.set_ylabel('Schizophrenia prevalence (%)', fontweight='bold')
ax.text(x_label_coord, y_label_coord, 'A', transform=ax.transAxes, size=24, weight='bold')
ax.set_title('Schizophrenia prevalence \nover time', fontweight='bold')
ax.legend()
ax = axes[1]

ax.plot(percentile_years, total_n_patients)
ax.vlines(2.958, 0, 160000, color = 'red', label = f'Dataset size: {len(temp_df)} patients')
ax.set_xlabel('Years since psychosis diagnosis', fontweight='bold')
ax.set_ylabel('Number of patients \nremaining in model', fontweight='bold')
ax.set_title('Number of patients in \nthe model over time', fontweight='bold')
ax.text(x_label_coord, y_label_coord, 'B', transform=ax.transAxes, size=24, weight='bold')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import matplotlib
import scipy.stats as stats
from sklearn.metrics import *
import sys
import seaborn as sns

sys.path.append('../')
from eval_utils import * 

dataset = 'mdcd_3yrs'
model_name = 'transformer_fullhistory_6mopred'
data_path = f'/data2/processed_datasets/ak4885/psychosis_schizophrenia_prediction/raw_data_{dataset}'
int_data_path = f'/data2/processed_datasets/ak4885/psychosis_schizophrenia_prediction/intermediate_data_{dataset}'
model_path = f'../skip_model_training/models/{model_name}'
featuretype = 'fullfeatures'
output_folder = f'figures_skiptrain_6mo_{featuretype}/'

test_output = pd.read_csv(f'{output_folder}/test_outputs.csv')
test_output = test_output.loc[test_output['days_since_start'] >= 365]


# Supplementary Figure 3: performance as a function of time since first visit
# Overall only
list_times_since_start = np.arange(365, np.percentile(test_output['days_since_start'], 90), 180)
print(list_times_since_start)

print(len(test_output))
list_eval_start_overall = calc_subtime_performances(test_output, list_times_since_start, 'days_since_start', cutoff_prob = None, y_pred_binary = test_output['y_binary_pred'])

list_prevs = []
for i in range(len(list_times_since_start)):
    temp_outputs = test_output.loc[(test_output['days_since_start'] <= list_times_since_start[i]) & (test_output['days_since_start'] > list_times_since_start[i-1])]
    list_prevs.append(temp_outputs['y_true'].sum()/len(temp_outputs))
print(list_prevs)
list_prevs = list_prevs[1:]

list_times_since_start = list_times_since_start[0:-1]/365
"""
metric = 'AUROC'
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16,7))
font = {'weight' : 'bold',
        'size'   : 22}
matplotlib.rc('font', **font)
x_label_coord = -0
y_label_coord = 1.02


plt.subplot(1,2,1)
ax = axes[0]
matplotlib.rc('font', **font)
plt.plot(list_times_since_start, [df.loc[metric, "mean"] for df in list_eval_start_overall], c = 'k')
plt.fill_between(list_times_since_start, [df.loc[metric, "CI_low"] for df in list_eval_start_overall], [df.loc[metric, "CI_high"] for df in list_eval_start_overall], color = 'k', alpha=0.5)
plt.xlabel('Years since first visit', fontdict=font)
plt.ylabel('AUROC', fontdict=font)
plt.xticks(fontsize=font["size"], fontweight=font["weight"])
plt.yticks(fontsize=font["size"], fontweight=font["weight"])
plt.title('Overall performance')
ax.text(x_label_coord, y_label_coord, 'A', transform=ax.transAxes, size=24, weight='bold')
ax.set_ylim([0.70, 0.85])

# SECOND SUBPLOT: LOOKING AT THE AMOUNT OF TIME THAT HAS PASSED SINCE START
plt.subplot(1,2,2)
ax = axes[1]
matplotlib.rc('font', **font)
plt.plot(list_times_since_start, 100*np.asarray(list_prevs))
plt.xlabel('Years since first visit', fontdict=font)
plt.ylabel('Schizophrenia prevalence (%)', fontdict=font)
plt.xticks(fontsize=font["size"], fontweight=font["weight"])
plt.yticks(fontsize=font["size"], fontweight=font["weight"])
plt.title('Schizophrenia prevalence \nanchored by first visit')
ax.text(x_label_coord, y_label_coord, 'B', transform=ax.transAxes, size=24, weight='bold')



plt.tight_layout()
plt.savefig(f'{output_folder}/auroc_over_time_since_start_nodemo.pdf', dpi = 300)